In [4]:
# Data handling
import numpy as np
import pandas as pd
import random
import re
import string


# Visualization
import matplotlib.pyplot as plt
import seaborn as sns



# Sklearn modules
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder

import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer


# ML Models
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb


# Others
import joblib

import nltk
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')

# Also useful to download WordNet for lemmatization
nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
#Vectorizer (TF-IDF, CountVectorizer, etc.) → Converts your cleaned text into numbers (features) that the model can understand.

#Model (LogisticRegression, RandomForest, etc.) → Learns patterns from those numerical features to predict Category, Priority, or Sentiment.

In [6]:
df = pd.read_csv('/content/E-Commerce_data - E-Commerce_data.csv')
df.columns

Index(['ticket_id', 'industry', 'category', 'priority', 'sentiment', 'title',
       'description', 'created_date', 'channel', 'resolution_time_hours'],
      dtype='object')

In [7]:
#remove duplication
df.drop_duplicates(subset=['title', 'description'], inplace=True)

In [8]:
#merge and drop old col
df['text'] = df['title'] + " " + df['description']
df.drop(['title', 'description'], axis=1, inplace=True)

In [9]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):

    # 1. Lowercase
    text = text.lower()

    # 2. Remove numbers
    text = re.sub(r'\d+', '', text)

    # 3. Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 4. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # 5. Tokenize
    words = text.split()

    # 6. Remove stopwords + Lemmatize #ed ing
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    return " ".join(words)

In [10]:
# apply cleanning
df['cleaned_text'] = df['text'].fillna('').apply(clean_text)

In [ ]:
#---------------------------------------------------------------CATEGORY

In [11]:
X = df['cleaned_text']
y_category = df['category']
#Train/Test Split
X_train_cat, X_test_cat, y_train_cat, y_test_cat = train_test_split(
    X, y_category, test_size=0.2, random_state=42
)


In [12]:
# train model
tfidf_lr_pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer()),
    ("classifier", LogisticRegression(max_iter=3000))
])

tfidf_lr_pipeline.fit(X_train_cat, y_train_cat)
pred_tfidf = tfidf_lr_pipeline.predict( X_test_cat)

print("TF-IDF + LR Accuracy:", accuracy_score(y_test_cat, pred_tfidf))
print(classification_report(y_test_cat, pred_tfidf))

TF-IDF + LR Accuracy: 0.9749857873791927
              precision    recall  f1-score   support

     account       0.98      0.97      0.98       383
     billing       0.97      0.98      0.98       501
    delivery       0.97      0.98      0.97       509
   technical       0.98      0.96      0.97       366

    accuracy                           0.97      1759
   macro avg       0.98      0.97      0.97      1759
weighted avg       0.98      0.97      0.97      1759



In [ ]:
joblib.dump(tfidf_lr_pipeline, 'category.joblib')

['category.joblib']

In [ ]:
#--------------------------------------------PRIORITY

In [13]:
X = df['cleaned_text']
y_priority = df['priority']
#Train/Test Split
X_train_pr, X_test_pr, y_train_pr, y_test_pr = train_test_split(
    X, y_priority, test_size=0.2, random_state=42
)



In [14]:
# train model
tfidf_lr_pipeline_pr = Pipeline([
    ("vectorizer", TfidfVectorizer()),
    ("classifier", LogisticRegression(max_iter=3000))
])

tfidf_lr_pipeline_pr.fit(X_train_pr, y_train_pr)
pred_tfidf = tfidf_lr_pipeline_pr.predict( X_test_pr)

print("TF-IDF + LR Accuracy:", accuracy_score(y_test_pr, pred_tfidf))
print(classification_report(y_test_pr,pred_tfidf))

TF-IDF + LR Accuracy: 0.6577600909607731
              precision    recall  f1-score   support

        high       0.71      0.83      0.76       876
         low       0.00      0.00      0.00       262
      medium       0.59      0.70      0.64       621

    accuracy                           0.66      1759
   macro avg       0.43      0.51      0.47      1759
weighted avg       0.56      0.66      0.61      1759



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
joblib.dump(tfidf_lr_pipeline_pr, 'priority.joblib')

['priority.joblib']

In [ ]:
#------------------------------------------Sentiment

In [15]:
pip install vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 9.8 MB/s eta 0:00:00


In [16]:
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

In [26]:
def get_sentiment_label(text):
    # TextBlob polarity
    tb = TextBlob(text)
    tb_polarity = tb.sentiment.polarity  # -1 to 1
    tb_subjectivity = tb.sentiment.subjectivity  # 0 to 1

    # VADER score
    vs = analyzer.polarity_scores(text)
    compound = vs['compound']

    # Combine rules
    if compound >= 0.3:
        return "Positive"
    elif compound <= -0.3:
        return "Negative"
    else:
        if tb_polarity > 0.4:
            return "Positive"
        elif tb_polarity < -0.4:
            return "Negative"
        else:
            return "Neutral"

In [27]:
#Apply
df['sentiment'] = df['cleaned_text'].apply(get_sentiment_label)


In [28]:
#split
X = df['cleaned_text']
y = df['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#train
pipeline_sentiment = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000)),
    ('clf', LogisticRegression(max_iter=3000))
])
pipeline_sentiment.fit(X_train, y_train)

# Make predictions using the sentiment pipeline on the sentiment test set
pred_sentiment = pipeline_sentiment.predict(X_test)

print("TF-IDF + LR Accuracy:", accuracy_score(y_test, pred_sentiment))
print(classification_report(y_test, pred_sentiment))

TF-IDF + LR Accuracy: 0.8959636156907334
              precision    recall  f1-score   support

    Negative       0.88      0.89      0.88       470
     Neutral       0.88      0.83      0.85       562
    Positive       0.92      0.96      0.94       727

    accuracy                           0.90      1759
   macro avg       0.89      0.89      0.89      1759
weighted avg       0.90      0.90      0.90      1759



In [29]:
joblib.dump(pipeline_sentiment, 'sentiment_pipeline.joblib')

['sentiment_pipeline.joblib']

In [ ]:
#suggested solution
import ollama

In [ ]:
def generate_ticket_response(text,category,priority,sentiment):
  """
  Generate professional telecom support response using olllama
  """
  system_prompt ="""
  you are a professional telecom support assistant.
  Guidelines:
  -respond professionally and concisely
  -show empathy if sentiment is negative
  -if priority is high,acknowledge urgency
  -do not invent policies or technical details
  -if information is missing,politely ask for clarification
  -keep response under 15 words
  """
  User_prompt = f"""
  ticket details:
  category: {category}
  priority: {priority}
  sentiment: {sentiment}
  customer message: {text}
  """
  try:
    response = ollama.chat(
        model="llama3.2",
        messages=[
            {"role": "system", "content": system_prompt.strip()},
            {"role": "user", "content": User_prompt.strip()}],
        options={
            "temperature":0.5,
            "top_p":0.95,
            "max_tokens":128,
        }

    )
    return response["message"]["content"].strip()
  except Exception as e:
    print(f"Error generating ticket response: {e}")
    return "we are currently reviewing your issue"


In [ ]:
#python -m uvicorn app:app --reload
#http://127.0.0.1:8000/docs